# 5G Network Intelligence — Telecom Data and Pretraining Budget Ablation

## Hypothesis
Can 12 million audited mobile-traffic prefix hours materially improve DLPRB forecasts? Three predeclared arms isolate additional data and tripled source-pretraining compute. Prior four-collection transfer failed the material-gain gate; more bytes are not a quality claim.


In [ ]:
# Review Run All is safe. The dedicated hosted worker enables this flag.
RUN_TRAINING = False


## Temporal protocol and units
Two reused development folds: 335/383/443 and 395/443/503 fit/stop/score. Context 168h, horizons 1/6/24h, all 2,880 beams. Original hours 504 onward remain excluded. MAE uses DLPRB usage units; WAPE is total absolute error divided by observed absolute load. Unverified utilization-percent units must not be claimed.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil
PROJECT = Path.cwd()
if not (PROJECT/'configs').is_dir():
    PROJECT = PROJECT.parent
def verified_input(filename, digest):
    matches = [p for p in Path('/kaggle/input').rglob(filename)
               if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
    if len(matches) != 1:
        raise ValueError('Exactly one pinned input manifest required')
    source = matches[0]
    manifest = json.loads(source.read_text())
    for name, expected in manifest['files'].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or ':' in name or '\\' in name:
            raise ValueError('Unsafe input path')
        path = source.parent/name
        if not path.resolve().is_relative_to(source.parent.resolve()) or path.is_symlink():
            raise ValueError('Input path escapes manifest root')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('Input checksum differs: '+name)
    return source.parent, manifest


## Input provenance
Reuse the already-approved hosted original dataset; upload only public licensed Telecom prefixes and new code. Verify both manifests before importing code. The collection remains one sampling domain across shards. Whole constant/duplicate source segments are excluded; original zeros and scoring masks are unchanged. ODbL attribution/share-alike obligations apply to databases; code retains MIT.


In [ ]:
if RUN_TRAINING:
    assert Path('/kaggle/working').is_dir(), 'Hosted Kaggle worker only.'
    base_root, base = verified_input('source_manifest.json', '3069c57c9f64f809464ead2fae973fbaa35a86c3a46884e15cc693354eecab46')
    extra_root, extra = verified_input('telecom_transfer_manifest.json', '10cd970e8724ca5258b855a37e2b3ebbda41b6e80b9984ad83cb56335ba702ed')
    assert not extra['original_beam_payload_included']
    PROJECT = Path('/kaggle/working/telecom_transfer')
    PROJECT.mkdir(exist_ok=True)
    # Only reuse base data/reports; import all code from the new pinned supplement.
    for source_root, manifest, base_only in [(base_root, base, True), (extra_root, extra, False)]:
        for name in manifest['files']:
            if base_only and not name.startswith(('data/', 'reports/')):
                continue
            target = PROJECT/name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source_root/name, target)
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = extra['source_commit']
    print('Verified source:', extra['source_commit'])


## Three arms and causal preprocessing
Telecom matched: 20×65,536 windows (5,120 updates), compared with the completed existing-data control. Existing extended and Telecom extended: each 60×65,536 windows (15,360 updates). Each collection has equal aggregate sampling weight. Use past-context normalization and transfer only the temporal encoder. Discard source output heads; learn physical DLPRB on original fit prefixes. Target fine-tuning remains seed 17, max 100 epochs, patience 12.


In [ ]:
import yaml
from IPython.display import display
for arm in ['telecom_matched', 'existing_extended', 'telecom_extended']:
    path = PROJECT/'configs'/f'telecom_transfer_{arm}.yaml'
    if path.exists():
        display(yaml.safe_load(path.read_text(encoding='utf-8')))


## Hosted training and debugging
Run the fixed arm order, never select source checkpoints using scoring. Keep cold states, source checkpoints, optimizer/RNG and all results in Kaggle Output. Source losses are dimensionless; target losses use physical units despite historical `scaled` log names.


In [ ]:
if RUN_TRAINING:
    import subprocess
    (PROJECT/'reports').mkdir(exist_ok=True)
    with (PROJECT/'reports/telecom_transfer_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_telecom_transfer'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
    if TRAIN_EXIT:
        raise RuntimeError('Inspect saved log/checkpoints; do not silently repeat fitting.')
else:
    print('Review mode: no fitting.')


## Evaluation, acceptance and limitations
Reconstruct full-population MAE/WAPE, verify cold states, encoder isolation, optimizer budgets, stopping choices and numerical parity. At least 2% mean MAE gain with no horizon regression is only an initial research gate; 5% and robustness/uncertainty checks are required for a material improvement claim. Partial source overlap and GOTS-F calendar identity are not certified absent. Repeated development, unknown Vigo scaling and correlated domains limit generalization claims. No holdout reuse or automatic production promotion.


In [ ]:
# Review scores only; independent verification is a separate acceptance step.
import pandas as pd
for arm in ['telecom_matched', 'existing_extended', 'telecom_extended']:
    path = PROJECT/'reports'/f'telecom_transfer_{arm}_multi_source_pretrain_development.csv'
    if path.exists():
        frame = pd.read_csv(path)
        frame['WAPE (%)'] = 100*frame['wape']
        print(arm)
        display(frame)
